# Fraud Detection using Classification Algorithms

**Goal:** Predict whether a bank account application is fraudulent (`fraud_bool` = 1) or not (0).

**Algorithms used:**
1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Tree
4. Random Forest

**Metrics:** Precision, Recall, F1-score (for the fraud class)


## Step 1: Import Libraries
We import pandas for handling data, matplotlib for plots, and sklearn for the models and metrics.


In [13]:
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report


## Step 2: Load the Dataset
We read the CSV file and look at its size, the first few rows, and how many fraud cases there are.


In [4]:
df = pd.read_csv("dataset/Base.csv")

print("Shape:", df.shape)
print(df["fraud_bool"].value_counts())
df.head()


Shape: (1000000, 32)
fraud_bool
0    988971
1     11029
Name: count, dtype: int64


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


## Step 3: Take a Smaller Sample
The dataset has 1 million rows, so training (especially KNN) would be very slow.
We take 100,000 rows. `stratify` keeps the same fraud ratio as the full data.


In [5]:
df, _ = train_test_split(df, train_size=100000, stratify=df["fraud_bool"], random_state=42)
print("New shape:", df.shape)


New shape: (100000, 32)


## Step 4: Preprocessing
- **X** = input features, **y** = target (`fraud_bool`)
- Text columns are converted to 0/1 columns using **one-hot encoding** (`pd.get_dummies`).


In [7]:
X = df.drop("fraud_bool", axis=1)
y = df["fraud_bool"]

X = pd.get_dummies(X, drop_first=True)
print("Features after encoding:", X.shape[1])


Features after encoding: 47


## Step 5: Train-Test Split
80% of the data is used for training and 20% for testing.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)


Train: (80000, 47)  Test: (20000, 47)


## Step 6: Feature Scaling
Logistic Regression and KNN depend on the size of the numbers (KNN measures distance).
`StandardScaler` puts every feature on the same scale (mean 0, std 1).
We **fit only on training data** so the test data stays unseen.


In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## Step 7: Create the Models
`class_weight="balanced"` tells the model to pay more attention to the rare fraud class.
(KNN does not have this option.)


In [10]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=100, max_depth=10, random_state=42),
}


## Step 8: Train and Evaluate Each Model
For each model: train on training data → predict on test data → calculate Precision, Recall, F1.

- **Precision:** Of all accounts predicted as fraud, how many were actually fraud?
- **Recall:** Of all actual fraud accounts, how many did we catch?
- **F1-score:** Balance between precision and recall.


In [11]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
    })

    print("=====", name, "=====")
    print(classification_report(y_test, y_pred, zero_division=0))


===== Logistic Regression =====
              precision    recall  f1-score   support

           0       1.00      0.81      0.89     19779
           1       0.04      0.79      0.08       221

    accuracy                           0.81     20000
   macro avg       0.52      0.80      0.49     20000
weighted avg       0.99      0.81      0.88     20000

===== KNN =====
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     19779
           1       0.17      0.01      0.02       221

    accuracy                           0.99     20000
   macro avg       0.58      0.50      0.51     20000
weighted avg       0.98      0.99      0.98     20000

===== Decision Tree =====
              precision    recall  f1-score   support

           0       1.00      0.80      0.89     19779
           1       0.04      0.66      0.07       221

    accuracy                           0.80     20000
   macro avg       0.52      0.73      0.48     20000


## Step 9: Compare Results


In [12]:
results_df = pd.DataFrame(results)
results_df


,Model,Precision,Recall,F1 Score
0,Logistic Regression,0.044125,0.791855,0.083592
1,KNN,0.166667,0.009050,0.017167
2,Decision Tree,0.035671,0.660633,0.067687
3,Random Forest,0.084218,0.574661,0.146906


## Step 12: Which Model Performed Best and Worst?

| Model | Precision | Recall | F1 |
|---|---|---|---|
| Logistic Regression | 0.044 | 0.792 | 0.084 |
| KNN | 0.167 | 0.009 | 0.017 |
| Decision Tree | 0.036 | 0.661 | 0.068 |
| Random Forest | 0.084 | 0.575 | 0.147 |

**Best by F1: Random Forest (0.147).** It has the best balance between precision and recall.
**Worst by F1: KNN (0.017).** It caught only about 2 of the 221 frauds in the test set (recall 0.009).

**Other observations**
- **Highest recall:** Logistic Regression (0.79). It catches the most fraud, but about 95% of its fraud alerts are false alarms (precision 0.044).
- **Highest precision:** KNN (0.167), but it predicted fraud only ~12 times, so that number is unreliable and it is useless in practice.
- **Decision Tree** is slightly worse than Logistic Regression on F1 (0.068 vs 0.084).
- KNN fails because it has no `class_weight` option. With only ~1% fraud, the 5 nearest neighbours of almost any point are non-fraud.
